# Notebook 18: `poison-control` — Data Poisoning Scanner Prototype

## Why This Matters

Every ML pipeline that ingests external data is vulnerable to poisoning. Web-scraped datasets, third-party data vendors, crowdsourced annotations, synthetic data from foundation models — any of these can carry subtle poisoning that degrades performance or introduces backdoors. The problem is not hypothetical: ImageNet has been shown to contain mislabeled and semantically ambiguous examples at significant rates, and several open-source datasets have been found to contain adversarially manipulated examples.

The challenge: you need to catch poisoned samples *before* training, not after you deploy a compromised model. This requires scanning training data for statistical anomalies, suspicious influences, and activation outliers — a workflow that has no standardized tooling.

This notebook builds `poison-control`, a prototype data poisoning scanner that combines three independent signals:
1. **Influence functions** — which training examples most influence model predictions?
2. **Isolation Forest** — which examples are statistical outliers in embedding space?
3. **Label consistency** — which examples have labels that disagree with model predictions and nearest-neighbor labels?

The prototype integrates these into a single audit report with risk scores.

## Background

### Influence Functions

Koh and Liang (2017) *"Understanding Black-box Predictions via Influence Functions"* (ICML Best Paper) brought influence functions from robust statistics into ML. An influence function estimates how much a model's prediction on test point `z_test` would change if training point `z_train` were removed from the dataset and the model retrained. The key formula:

```
I(z_train, z_test) ≈ -∇L(z_test)ᵀ H⁻¹ ∇L(z_train)
```

where H is the Hessian of the training loss. Computing this exactly requires inverting the Hessian of a neural network — which is O(p²) for p parameters. In practice, we approximate using a few-step conjugate gradient or just use gradient similarity as a proxy.

For poisoning detection, we look for training examples with *abnormally high positive influence* on many test predictions — these are the examples the model relies on disproportionately, which is a red flag.

### Isolation Forest

Liu et al. (2008) introduced Isolation Forest for anomaly detection. The insight: anomalies are few and different. A random decision tree isolates anomalies in fewer splits than normal points. The isolation score averages the path length across many trees — shorter average path = more anomalous.

For poisoning detection, we extract penultimate-layer representations and run Isolation Forest. Poisoned examples often have anomalous embeddings — they're far from the manifold of clean examples in representation space. This is the same intuition as Activation Clustering (notebook 9), but generalized to any outlier, not just backdoor-triggered outliers.

### Label-Prediction Disagreement

A simple but powerful heuristic: if a model trained on the full dataset consistently predicts a different class for an example than its labeled class, that example is suspicious. High loss on training data after full training = potential mislabeling or poisoning.

Combined with k-NN label consistency (does the example's label agree with its nearest neighbors' labels in embedding space?), this gives a robust signal for label flipping and clean-label attacks.

### Combining Signals

Individual signals have high false positive rates. Influence functions flag examples that are unusual but might just be hard or rare. Isolation Forest flags statistical outliers that might be legitimate edge cases. Label disagreement flags high-loss examples that might just be genuinely ambiguous.

The key is combining signals: examples that are *simultaneously* high-influence AND activation outliers AND high-loss are much more likely to be genuine poisoning than examples flagged by only one detector.

## Structure of This Notebook

1. Create a poisoned dataset (label flipping + clean-label attack mix)
2. Signal 1: Approximate influence functions
3. Signal 2: Isolation Forest on embeddings
4. Signal 3: Label-prediction disagreement + k-NN label consistency
5. Ensemble scoring and ranked report
6. Precision/recall evaluation of the combined scanner

## What You'll Learn

- How influence functions identify high-impact training samples
- Why Isolation Forest works in embedding space for anomaly detection
- How to combine multiple weak signals into a stronger composite detector
- The precision/recall tradeoff in data poisoning scanning
- How to build an audit-ready risk report

In [ ]:
%pip install torch torchvision numpy matplotlib scikit-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import precision_recall_curve, average_precision_score
import copy

device = (
    torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cuda') if torch.cuda.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')
torch.manual_seed(42)
np.random.seed(42)

## 1. Create Poisoned Dataset

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)

    def get_embeddings(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        return F.relu(self.fc1(x))


transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
full_train = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)

# Use 5000 training samples for speed
N_TRAIN = 5000
POISON_RATE = 0.05  # 5% poisoned
N_POISON = int(N_TRAIN * POISON_RATE)

# Extract subset
train_imgs = torch.stack([full_train[i][0] for i in range(N_TRAIN)])
train_labels = torch.tensor([full_train[i][1] for i in range(N_TRAIN)])

# Create poisoned version: flip labels on random subset
poison_indices = np.random.choice(N_TRAIN, N_POISON, replace=False)
clean_mask = np.ones(N_TRAIN, dtype=bool)
clean_mask[poison_indices] = False

poisoned_labels = train_labels.clone()
# Random label flipping (source_class → random other class)
for idx in poison_indices:
    original = poisoned_labels[idx].item()
    new_label = np.random.choice([c for c in range(10) if c != original])
    poisoned_labels[idx] = new_label

# Ground truth poison mask for evaluation
is_poisoned = ~clean_mask  # True if poisoned

print(f'Dataset: {N_TRAIN} total, {N_POISON} poisoned ({POISON_RATE*100:.0f}%)')
print(f'Poisoned indices (first 10): {sorted(poison_indices)[:10]}')

# Train model on poisoned data
poisoned_dataset = TensorDataset(train_imgs, poisoned_labels)
train_loader = DataLoader(poisoned_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

model = MnistCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

print('\nTraining on poisoned dataset...')
for epoch in range(5):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        F.cross_entropy(model(x), y).backward()
        opt.step()

model.eval()
correct = sum(
    (model(x.to(device)).argmax(1) == y.to(device)).sum().item()
    for x, y in test_loader
)
print(f'Model accuracy (trained on poisoned data): {correct/len(test_dataset):.4f}')

## 2. Signal 1 — Approximate Influence Functions

In [ ]:
def compute_gradient(model, x, y):
    """Compute flattened gradient of loss w.r.t. model parameters."""
    model.zero_grad()
    loss = F.cross_entropy(model(x.unsqueeze(0).to(device)), y.unsqueeze(0).to(device))
    loss.backward()
    grads = [p.grad.flatten() for p in model.parameters() if p.grad is not None]
    return torch.cat(grads).detach()


def approximate_influence_scores(
    model,
    train_imgs,
    train_labels,
    n_test_samples: int = 100,
    n_train_samples: int = 500,
):
    """
    Approximate influence of each training sample using gradient similarity.

    True influence: -∇L(z_test)ᵀ H⁻¹ ∇L(z_train)
    Approximation (TracIn proxy): ∇L(z_test)ᵀ ∇L(z_train)

    High positive influence = training sample helps the model predict correctly.
    High absolute influence = training sample has large effect on model behavior.
    Very high absolute influence compared to others = suspicious.
    """
    model.eval()

    # Sample test points
    test_subset = Subset(test_dataset, range(n_test_samples))
    test_grads = torch.stack([
        compute_gradient(model,
                         test_subset[i][0],
                         torch.tensor(test_subset[i][1]))
        for i in range(n_test_samples)
    ])  # (n_test, n_params)

    # Sample training points
    train_sample_idx = np.random.choice(len(train_imgs), n_train_samples, replace=False)
    train_grads = torch.stack([
        compute_gradient(model, train_imgs[i], train_labels[i])
        for i in train_sample_idx
    ])  # (n_train_sample, n_params)

    # Influence = average gradient dot product over test set
    # (n_train_sample, n_test) @ (n_test, 1) → (n_train_sample,)
    influence_matrix = train_grads @ test_grads.T  # (n_train, n_test)
    avg_influence = influence_matrix.abs().mean(dim=1)  # mean abs influence

    # Map back to full train set (others get 0)
    full_scores = np.zeros(len(train_imgs))
    for i, idx in enumerate(train_sample_idx):
        full_scores[idx] = avg_influence[i].item()

    return full_scores, train_sample_idx


print('Computing approximate influence scores (100 test, 500 train samples)...')
influence_scores, sampled_train_idx = approximate_influence_scores(
    model, train_imgs, poisoned_labels,
    n_test_samples=100, n_train_samples=500
)

# Z-score normalization
inf_mean = influence_scores[sampled_train_idx].mean()
inf_std = influence_scores[sampled_train_idx].std()
influence_z = (influence_scores - inf_mean) / (inf_std + 1e-8)

print(f'Influence scores — mean: {inf_mean:.4f}, std: {inf_std:.4f}')
print(f'Top-10 high-influence training samples: {np.argsort(-influence_scores)[:10]}')

## 3. Signal 2 — Isolation Forest on Embeddings

In [ ]:
def extract_embeddings(model, imgs, batch_size=256):
    """Extract penultimate-layer (128-d) embeddings for all training images."""
    model.eval()
    embeds = []
    with torch.no_grad():
        for i in range(0, len(imgs), batch_size):
            batch = imgs[i:i+batch_size].to(device)
            embeds.append(model.get_embeddings(batch).cpu())
    return torch.cat(embeds).numpy()


def isolation_forest_scores(embeddings, contamination=0.1):
    """
    Fit Isolation Forest on embeddings.
    Returns anomaly score for each sample:
    higher score = more anomalous (more likely poisoned).

    sklearn's decision_function returns: negative anomaly score
    (more negative = more anomalous). We negate it so higher = worse.
    """
    clf = IsolationForest(n_estimators=100, contamination=contamination, random_state=42)
    clf.fit(embeddings)
    scores = -clf.decision_function(embeddings)  # negate: higher = more anomalous
    return scores


print('Extracting embeddings...')
embeddings = extract_embeddings(model, train_imgs)
print(f'Embeddings shape: {embeddings.shape}')

print('Running Isolation Forest...')
iso_scores = isolation_forest_scores(embeddings, contamination=POISON_RATE)
print(f'Isolation scores — mean: {iso_scores.mean():.4f}, std: {iso_scores.std():.4f}')

# Visualize: PCA of embeddings colored by Isolation Forest score
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
embed_2d = pca.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

scatter1 = axes[0].scatter(embed_2d[:, 0], embed_2d[:, 1],
                            c=iso_scores, cmap='RdYlGn_r', s=5, alpha=0.6)
plt.colorbar(scatter1, ax=axes[0])
axes[0].set_title('Embeddings — Isolation Forest Score')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

# Color by true poison status
colors = ['red' if p else 'steelblue' for p in is_poisoned]
sizes = [15 if p else 3 for p in is_poisoned]
axes[1].scatter(embed_2d[:, 0], embed_2d[:, 1], c=colors, s=sizes, alpha=0.5)
axes[1].scatter([], [], color='red', s=30, label='Poisoned (ground truth)')
axes[1].scatter([], [], color='steelblue', s=5, label='Clean')
axes[1].legend()
axes[1].set_title('Embeddings — True Poison Status')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

## 4. Signal 3 — Label-Prediction Disagreement and k-NN Consistency

In [ ]:
def label_disagreement_scores(model, imgs, labels, batch_size=256):
    """
    Compute per-sample training loss (after full training).
    High loss = model can't fit this sample = suspicious.

    Also compute binary disagreement: 1 if predicted class ≠ labeled class.
    """
    model.eval()
    losses = []
    disagrees = []

    with torch.no_grad():
        for i in range(0, len(imgs), batch_size):
            x = imgs[i:i+batch_size].to(device)
            y = labels[i:i+batch_size].to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y, reduction='none')
            preds = logits.argmax(1)
            losses.append(loss.cpu())
            disagrees.append((preds != y).cpu())

    losses = torch.cat(losses).numpy()
    disagrees = torch.cat(disagrees).numpy().astype(float)
    return losses, disagrees


def knn_label_consistency(embeddings, labels, k=5):
    """
    For each training sample, check if its label agrees with its k nearest
    neighbors' labels (in embedding space).

    Inconsistency score = fraction of k neighbors with different label.
    High inconsistency = the label doesn't match the local neighborhood = suspicious.
    """
    nbrs = NearestNeighbors(n_neighbors=k+1, metric='euclidean').fit(embeddings)
    distances, indices = nbrs.kneighbors(embeddings)

    inconsistency = np.zeros(len(labels))
    for i in range(len(labels)):
        neighbor_idx = indices[i, 1:]  # exclude self
        neighbor_labels = labels[neighbor_idx]
        inconsistency[i] = (neighbor_labels != labels[i]).mean()

    return inconsistency


print('Computing label-prediction disagreement...')
train_losses, train_disagrees = label_disagreement_scores(
    model, train_imgs, poisoned_labels
)

print('Computing k-NN label consistency (k=5)...')
knn_inconsistency = knn_label_consistency(embeddings, poisoned_labels.numpy(), k=5)

print(f'Mean training loss — clean: {train_losses[clean_mask].mean():.4f}, '
      f'poisoned: {train_losses[is_poisoned].mean():.4f}')
print(f'k-NN inconsistency — clean: {knn_inconsistency[clean_mask].mean():.4f}, '
      f'poisoned: {knn_inconsistency[is_poisoned].mean():.4f}')

## 5. Ensemble Scoring and Audit Report

In [ ]:
def normalize_01(scores):
    """Normalize scores to [0, 1] range."""
    mn, mx = scores.min(), scores.max()
    return (scores - mn) / (mx - mn + 1e-8)


def poison_control_scan(
    model,
    train_imgs,
    train_labels,
    n_test_samples: int = 100,
    n_influence_samples: int = 300,
    knn_k: int = 5,
    contamination: float = 0.1,
    weights=(0.3, 0.4, 0.3),  # (influence, isolation_forest, label_disagreement)
):
    """
    poison-control: Complete data poisoning scanner.

    Runs three independent detectors and combines them into a
    composite risk score. Higher score = more suspicious.

    Returns:
        risk_scores: np.array of shape (n_train,) — composite risk score
        signals: dict with individual signal arrays
        report: human-readable audit summary
    """
    print('[poison-control] Starting scan...')

    # Signal 1: Influence
    print('  [1/3] Computing influence scores...')
    inf_scores, _ = approximate_influence_scores(
        model, train_imgs, train_labels,
        n_test_samples=n_test_samples,
        n_train_samples=n_influence_samples
    )
    inf_norm = normalize_01(inf_scores)

    # Signal 2: Isolation Forest
    print('  [2/3] Running Isolation Forest on embeddings...')
    embeds = extract_embeddings(model, train_imgs)
    iso = isolation_forest_scores(embeds, contamination=contamination)
    iso_norm = normalize_01(iso)

    # Signal 3: Label disagreement + k-NN consistency
    print('  [3/3] Checking label consistency...')
    losses, disagrees = label_disagreement_scores(model, train_imgs, train_labels)
    knn_inc = knn_label_consistency(embeds, train_labels.numpy(), k=knn_k)
    label_signal = normalize_01(losses) * 0.5 + normalize_01(knn_inc) * 0.5

    # Ensemble
    w1, w2, w3 = weights
    composite = w1 * inf_norm + w2 * iso_norm + w3 * label_signal
    composite_norm = normalize_01(composite)

    signals = {
        'influence': inf_norm,
        'isolation_forest': iso_norm,
        'label_disagreement': label_signal,
        'composite': composite_norm,
    }

    print('[poison-control] Scan complete.')
    return composite_norm, signals


print('Running poison-control scan...')
risk_scores, signals = poison_control_scan(
    model, train_imgs, poisoned_labels,
    n_test_samples=80, n_influence_samples=300,
    contamination=POISON_RATE
)

# Top-K flagged
TOP_K = N_POISON * 2  # flag 2x the true poison count (recall-focused)
top_flagged = np.argsort(-risk_scores)[:TOP_K]
true_positives = sum(is_poisoned[i] for i in top_flagged)
precision = true_positives / TOP_K
recall = true_positives / N_POISON

print(f'\n=== poison-control Audit Report ===')
print(f'Dataset: {N_TRAIN} samples, suspected poison rate: {contamination:.0%}')
print(f'Top-{TOP_K} flagged samples:')
print(f'  True poisoned: {true_positives}/{N_POISON}')
print(f'  Precision: {precision:.4f}')
print(f'  Recall: {recall:.4f}')

risk_level = 'HIGH' if recall > 0.7 else 'MEDIUM' if recall > 0.4 else 'LOW'
print(f'  Risk level: {risk_level}')

## 6. Precision-Recall Evaluation

In [ ]:
y_true = is_poisoned.astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PR curves for each signal
for signal_name, signal_scores in signals.items():
    p, r, _ = precision_recall_curve(y_true, signal_scores)
    ap = average_precision_score(y_true, signal_scores)
    axes[0].plot(r, p, label=f'{signal_name} (AP={ap:.3f})')

baseline_precision = y_true.mean()
axes[0].axhline(baseline_precision, color='gray', linestyle='--',
                label=f'Random (precision={baseline_precision:.3f})')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curves: Poison Detection')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Score distribution: clean vs. poisoned
axes[1].hist(risk_scores[clean_mask], bins=40, alpha=0.6, color='steelblue',
             density=True, label='Clean samples')
axes[1].hist(risk_scores[is_poisoned], bins=20, alpha=0.6, color='red',
             density=True, label='Poisoned samples')
axes[1].set_xlabel('Composite Risk Score')
axes[1].set_ylabel('Density')
axes[1].set_title('Risk Score Distribution: Clean vs. Poisoned')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('poison-control: Scanner Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Signal correlation
print('Signal correlation with true poison labels:')
for name, scores in signals.items():
    corr = np.corrcoef(scores, y_true)[0, 1]
    ap = average_precision_score(y_true, scores)
    print(f'  {name:<25}: corr={corr:.4f}, AP={ap:.4f}')

In [ ]:
# Final audit report
def generate_audit_report(risk_scores, train_labels, poisoned_labels_actual, threshold_percentile=90):
    """Generate a human-readable audit report."""
    threshold = np.percentile(risk_scores, threshold_percentile)
    flagged = np.where(risk_scores >= threshold)[0]

    print('=' * 60)
    print('          POISON-CONTROL AUDIT REPORT')
    print('=' * 60)
    print(f'Total training samples: {len(risk_scores)}')
    print(f'Flag threshold: {threshold_percentile}th percentile (score >= {threshold:.4f})')
    print(f'Flagged samples: {len(flagged)} ({len(flagged)/len(risk_scores)*100:.1f}%)')
    print()
    print('Top 10 Most Suspicious Samples:')
    print(f'{"Index":>6} {"Risk":>7} {"Given Label":>12} {"Model Pred":>12}')
    print('-' * 45)

    model.eval()
    top_10 = np.argsort(-risk_scores)[:10]
    with torch.no_grad():
        preds = model(train_imgs[top_10].to(device)).argmax(1).cpu()

    for i, (idx, pred) in enumerate(zip(top_10, preds)):
        given = poisoned_labels_actual[idx].item()
        flag = '*** POISON ***' if is_poisoned[idx] else ''
        print(f'{idx:>6} {risk_scores[idx]:>7.4f} {given:>12} {pred.item():>12}  {flag}')

    print()
    print('Recommended Actions:')
    if len(flagged) / len(risk_scores) > 0.15:
        print('  [HIGH RISK] Remove flagged samples and retrain')
    elif len(flagged) / len(risk_scores) > 0.05:
        print('  [MEDIUM RISK] Manually review flagged samples')
    else:
        print('  [LOW RISK] Minor anomalies detected — spot check recommended')


generate_audit_report(risk_scores, train_labels, poisoned_labels)

## Summary

### What We Built

| Component | Signal | Detection Method |
|-----------|--------|-----------------|
| `approximate_influence_scores()` | Gradient similarity proxy | High influence → model depends heavily on this point |
| `isolation_forest_scores()` | Embedding anomaly | Short isolation path → rare in representation space |
| `label_disagreement_scores()` | Training loss | High loss after full training → model can't fit this sample |
| `knn_label_consistency()` | k-NN label agreement | Neighbor labels differ → label doesn't fit local manifold |
| `poison_control_scan()` | Ensemble of 4 signals | Weighted combination into composite risk score |
| `generate_audit_report()` | Ranked report | Human-readable with top suspects and recommendations |

### Key Takeaways

- **No single signal reliably catches all poisoning.** Influence functions miss samples with average gradient impact; Isolation Forest misses systematic poisoning that shifts the distribution; loss heuristics miss well-crafted clean-label attacks.
- **Ensemble scanning dramatically improves detection.** Combining signals catches samples that any single detector would miss, while reducing false positives by requiring multiple signals to fire simultaneously.
- **Label flipping is the easiest to detect.** Flipped labels create high training loss and k-NN inconsistency — the label doesn't fit the local neighborhood. Clean-label attacks (which don't change labels) are much harder to catch.
- **The detector must run on a trusted model.** Using the poisoned model's embeddings to detect poisoning is circular — but in practice, even a slightly-poisoned model's representations are diagnostic enough to flag obvious poisoning.
- **Human review is essential.** The scanner narrows the suspect pool; a human expert reviews the flagged samples. The goal is to go from "check 5000 samples" to "check 50 suspects."

### Production Hardening

A production `poison-control` tool would add:
- Cleaner influence function computation using conjugate gradient Hessian inversion
- SPECTRE (Hayase et al., 2021) for clean-label attack detection
- Spectral signatures (Tran et al., 2018) for backdoor-specific detection
- Per-class anomaly analysis (poisoning is often class-targeted)
- Integration with dataset versioning (detect when poisoning was introduced)

Next: Notebook 19 builds `model-scan` — a behavioral backdoor scanner that analyzes a trained model's decision boundaries and activation patterns to detect whether a model has been backdoored, without access to the training data.